# 32 — Boosting candidate

Entrena un `HistGradientBoostingClassifier` sobre una muestra TRAIN determinística y acotada por memoria; evalúa sobre VALIDATION completo. TEST no se consume.

In [1]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m3_gate = paths.docs / "decisions" / "M3_GATE.md"
assert m3_gate.exists() and "M3 ✅ CLOSED" in m3_gate.read_text(encoding="utf-8"), (
    "M3 no está formalmente cerrado. No iniciar Modeling."
)
GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
MODEL_DIR = paths.root / "artifacts" / "models" / "m4"
MANIFEST_DIR = paths.root / "artifacts" / "manifests"

{
    "gold_schema": GOLD_SPEC.schema_version,
    "modeling_schema": MODEL_SPEC.schema_version,
    "target": MODEL_SPEC.target,
    "model_matrix": MODEL_SPEC.model_matrix,
    "operational_review_fraction": MODEL_SPEC.ranking.operational_fraction,
}

assert (paths.reports / "modeling" / "31_validation.parquet").exists(), "Ejecuta 31 primero."
from argos.models.boosting import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION


'm4-32-hist-gradient-boosting-v1'

In [2]:
from argos.models.boosting import train_boosting_model

result = train_boosting_model(
    root=paths.root,
    gold_spec=GOLD_SPEC,
    modeling_spec=MODEL_SPEC,
    reports_root=paths.reports,
    model_dir=MODEL_DIR,
    manifest_dir=MANIFEST_DIR,
)
result.metrics


,model,split,rows,positives,prevalence,pr_auc,roc_auc,log_loss,brier,ece
0,hist_gradient_boosting,validation,950670,9180,0.009656,0.064529,0.853849,0.225765,0.063614,0.144308


In [3]:
result.ranking

,model,split,review_fraction,reviewed,captured_positives,precision_at_k,recall_at_k,lift_at_k,cumulative_gain
0,hist_gradient_boosting,validation,0.001,951,19,0.019979,0.002070,2.068999,0.002070
1,hist_gradient_boosting,validation,0.005,4754,307,0.064577,0.033442,6.687539,0.033442
2,hist_gradient_boosting,validation,0.010,9507,1025,0.107815,0.111656,11.165225,0.111656
3,hist_gradient_boosting,validation,0.025,23767,2401,0.101022,0.261547,10.461764,0.261547
4,hist_gradient_boosting,validation,0.050,47534,3665,0.077103,0.399237,7.984665,0.399237
5,hist_gradient_boosting,validation,0.100,95067,5504,0.057896,0.599564,5.995643,0.599564


In [4]:
result.validation

,check_id,category,blocking,passed,observed,requirement
0,boosting_train_sample,training,True,True,"{""cap"": 750000, ""sample_rows"": 750000, ""train_...",boosting uses a deterministic TRAIN-only sampl...
1,boosting_validation_population,evaluation,True,True,"{""distinct_contracts"": 950670, ""expected"": 950...",boosting validation predictions cover the comp...
2,boosting_probabilities_finite,quality,True,True,"{""max"": 0.994472497764912, ""min"": 0.0003338607...",boosting validation scores are finite probabil...
3,boosting_test_isolation,test_isolation,True,True,"{""consumed_splits"": [""train"", ""validation""]}",notebook 32 does not consume TEST rows


In [5]:
blocking = result.validation.loc[result.validation["blocking"]]
print("32 ✅ PASS" if result.ready_for_next_notebook else "32 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Model:", result.model_path)
assert result.ready_for_next_notebook


32 ✅ PASS
Blocking checks: 4/4
Model: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/models/m4/hist_gradient_boosting.joblib
